In [ ]:
import importlib.util
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, HTML
import subprocess
import sys
import time

# Package Verification & Installation Setup
packages_to_check = [
    "optuna",
    "tqdm",
    "gdown",
    "catboost",
    "scikit-learn",
]

missing_packages = [
    pkg for pkg in packages_to_check
    if importlib.util.find_spec(pkg) is None
]

progress = widgets.IntProgress(
    value=0,
    min=0,
    max=100,
    description="Setup:",
    bar_style="info"
)

status = widgets.HTML(value="Starting setup...")
display(progress, status)

progress.value = 20
status.value = "Checking required packages..."

if missing_packages:
    step_val = 70 / len(missing_packages)

    for i, pkg in enumerate(missing_packages):
        status.value = f"Installing package: {pkg}..."

        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", pkg],
            check=True
        )

        progress.value = int(20 + (i + 1) * step_val)

progress.value = 100
progress.bar_style = "success"
status.value = "✓ All required packages are available."

# Imports
import numpy as np
import optuna
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.notebook import tqdm

from catboost import CatBoostClassifier

from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

optuna.logging.set_verbosity(optuna.logging.WARNING)

## 1. Load the data and hold out a test set

The group's EDA notebook (`eda.ipynb`) cleans the raw data and saves the result as `data/data_clean.csv`. The cleaning steps, and how many rows each one removed, are documented there.

> The `data/` folder is **not** in the repository. Download the dataset from the Drive link in the README and run the EDA notebook first, because that creates this file. The eda.ipynb also has the download built-in the code, so it can run, download the file and create the data folder conveniently.

We then split off **20% as a test set** and do not touch it again until the champion is chosen. Everything in between (cross-validation, grid search, Hyperopt) happens on the training 80% only.

| Set | Used for |
|---|---|
| **Train (80%)** | Cross-validation, model comparison, hyperparameter tuning |
| **Test (20%)** | One final check of the champion on data no model has ever seen |

`stratify=y` keeps the same class proportions in both halves.

> **Reproducibility:** every step that involves randomness uses `random_state=42`, so anyone who runs this notebook gets the same champion and the same model file.

In [ ]:
# Settings
RANDOM_STATE = 42       # the same number everywhere, so results are reproducible
N_SPLITS = 5            # 5-fold cross-validation
TEST_SIZE = 0.2         # 20% of the data is held back and never used for tuning
SCORING = "f1_macro"    # our comparison metric — see section 3 for why
MAX_EVALS = 10          # how many settings Hyperopt tries per model

# StratifiedKFold keeps the class proportions the same in every fold.
# That matters here because our classes are imbalanced.
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# --- The five OCEAN traits (column names start with their letter) ---------
TRAITS = {
    "O": "Openness to Experience",
    "C": "Conscientiousness",
    "E": "Extraversion",
    "A": "Agreeableness",
    "N": "Neuroticism"
}

ocean_order = list(TRAITS.values())

# --- One fixed colour per personality type, used in every plot -------------
TARGET_COLORS = {
    "Resilient":        "#e07a5f",  # muted terracotta/coral
    "Overcontroller":   "#81b29a",  # sage/mint green
    "Undercontroller":  "#f2cc8f",  # warm ochre yellow
    "Moderate":         "#2b5c6f"   # petroleum blue (same as the primary theme colour)
}

target_order = list(TARGET_COLORS.keys())

# --- Colours for titles, info boxes and heatmaps -----------------------------
THEME = {
    "primary":          "#2b5c6f",  # Main petroleum blue for titles & headers
    "success_bg":       "#edf7f4",  # Soft mint/sage background
    "success_border":   "#81b29a",  # Sage/mint green border (matches Overcontroller)
    "danger_bg":        "#fdf2f2",  # Soft coral/red background
    "danger_border":    "#e07a5f",  # Terracotta/coral border (matches Resilient)
    "palette":          "Blues"
}

In [ ]:
# 1. Load data
DATA_FILE = Path("DATA") / "data_clean.csv"
df = pd.read_csv(DATA_FILE)

X = df.drop(columns="target")   # features: 19 answers + age + gender + hand
y = df["target"]                # what we predict, kept as readable text

# the 19 questionnaire columns = everything except the demographics and the target
item_cols = [c for c in df.columns if c not in ["age", "gender", "hand", "target"]]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,                 # same class mix in both halves
)

display(HTML(f"""
<table style="background:{THEME['success_bg']}; border-left:4px solid {THEME['success_border']}; 
              border-collapse:collapse; margin:8px 0; font-size:14px; text-align:left;">
    <tr>
        <td style="padding:10px 14px; border:none;" colspan="2"><b>Total:</b> {len(df):,} rows</td>
    </tr>
    <tr>
        <td style="padding:4px 14px; border:none;"><b>Train:</b> {len(X_train):,} rows</td>
        <td style="padding:4px 14px; border:none;"><b>Test:</b> {len(X_test):,} rows</td>
    </tr>
    <tr>
        <td style="padding:4px 14px 10px 14px; border:none;"><b>Features:</b> {X.shape[1]}</td>
        <td style="padding:4px 14px 10px 14px; border:none;"><b>Items:</b> {len(item_cols)}</td>
    </tr>
</table>
"""))

df.head()

In [ ]:
counts = y.value_counts()

target_overview = pd.DataFrame({"count": counts,"share_%": (counts / len(y) * 100).round(1)})
target_overview.index.name = "personality type"
display(target_overview)

plt.figure(figsize=(7, 3.5))
sns.countplot(x=y, order=target_order, hue=y, palette=TARGET_COLORS, legend=False)
plt.title("How many people of each personality type?", color=THEME["primary"])
plt.xlabel("")
plt.ylabel("")
plt.show()

## 2. The preprocessing pipeline

Models need numbers, with no gaps and on a comparable scale. Our columns need different
treatment:

- **numbers** (the 19 items + `age`): fill any gaps with the median, then scale
- **categories** (`gender`, `hand`): fill any gaps with the most frequent value, then one-hot encode

A `ColumnTransformer` made of two small pipelines describes this once. Every model below
reuses it, so each model sees exactly the same prepared data and the comparison is fair.

The target stays as **text** (`"Resilient"`), so predictions come out readable in the app —
no label encoder needed.

### What the preprocessor actually does

`gender` and `hand` turn into 0/1 columns, and no gaps are left behind. We fit it on the
**training data only** here, just to look at it — inside the pipelines below it is refit
within every fold, so nothing leaks from a validation fold into the scaling.

In [ ]:
# 2. Column definitions
question_cols = [
    'N1', 'N2', 'N3', 'N4', 'N5', 'N6', 'N7', 'N8', 'N9', 'N10',
    'E1', 'E3', 'E4', 'E5', 'E7', 'E9', 'E10', 'C4', 'A4'
]
numeric_cols = question_cols + ['age']
categorical_cols = ['gender', 'hand']

# 2.5 Preprocessing
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
]) 

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
]) 

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

display(preprocessor)

In [ ]:
# 3.1. Pipeline for Logistic Regression
pipe_logreg = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

# 3.2. Pipeline for KNN
pipe_knn = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", KNeighborsClassifier()),
    ]
)

# 3.3. Pipeline for Random Forest
pipe_rf = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(random_state=42)),
    ]
)

# 3.4a. Pipeline for Hist Gradient Boosting
pipe_hgb = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", HistGradientBoostingClassifier(random_state=42)),
    ]
)

# 3.4b. Pipeline for Hist Gradient Boosting (Tuned variant)
pipe_hgb_tuned = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", HistGradientBoostingClassifier(random_state=42)),
    ]
)

# 3.5. Pipeline for CatBoost
pipe_cat = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", CatBoostClassifier(random_state=42, verbose=0)),
    ]
)

df_pipelines = pd.DataFrame([
    {
        "Pipeline": "<code>pipe_logreg</code>",
        "Model": "LogisticRegression",
        "Family": "Linear Model",
        "Why?": "Simple, highly interpretable baseline."
    },
    {
        "Pipeline": "<code>pipe_knn</code>",
        "Model": "KNeighborsClassifier",
        "Family": "Distance-based",
        "Why?": "Classifies based on similar answers; scaling is important."
    },
    {
        "Pipeline": "<code>pipe_rf</code>",
        "Model": "RandomForestClassifier",
        "Family": "Bagging / Decision Trees",
        "Why?": "Captures non-linear relationships and feature interactions."
    },
    {
        "Pipeline": "<code>pipe_hgb</code>",
        "Model": "HistGradientBoostingClassifier",
        "Family": "Boosting / Decision Trees",
        "Why?": "Histogram-based boosting as a fast candidate for tabular data."
    },
    {
        "Pipeline": "<code>pipe_hgb_tuned</code>",
        "Model": "HistGradientBoostingClassifier",
        "Family": "Boosting / Decision Trees",
        "Why?": "Optimized variant of HistGradientBoosting for performance gains."
    },
    {
        "Pipeline": "<code>pipe_cat</code>",
        "Model": "CatBoostClassifier",
        "Family": "Boosting / Decision Trees",
        "Why?": "Robust boosting method, strong performance on structured data."
    }
])

html_table = f"""
<table style="width:100%; border-collapse:collapse; margin:12px 0; font-size:14px; font-family:sans-serif; text-align:left;">
    <thead>
        <tr style="background:{THEME['success_bg']}; border-bottom:2px solid {THEME['success_border']};">
            <th style="padding:10px 14px; color:{THEME['primary']};">Pipeline</th>
            <th style="padding:10px 14px; color:{THEME['primary']};">Model</th>
            <th style="padding:10px 14px; color:{THEME['primary']};">Family</th>
            <th style="padding:10px 14px; color:{THEME['primary']};">Why?</th>
        </tr>
    </thead>
    <tbody>
"""

for i, row in df_pipelines.iterrows():
    bg_color = "#fcfdfd" if i % 2 == 0 else f"{THEME['success_bg']}55"
    html_table += f"""
        <tr style="background:{bg_color}; border-bottom:1px solid #eaeaea;">
            <td style="padding:10px 14px;">{row['Pipeline']}</td>
            <td style="padding:10px 14px;">{row['Model']}</td>
            <td style="padding:10px 14px;">{row['Family']}</td>
            <td style="padding:10px 14px;">{row['Why?']}</td>
        </tr>
    """

html_table += """
    </tbody>
</table>
"""

display(HTML(html_table))

display(pipe_hgb)   # show one pipeline as an example: the shared preprocessor + the classifier

In [ ]:
HTML FOLDS KOMPAKT + BATHCHART + SCHNELL GRÜN LANGSAM ROT

## 6. Hyperparameter tuning — all four models

A hyperparameter is a setting we choose before training (how deep a tree may grow, how strong
the regularisation is). Tuning means searching for the combination that scores best.

The brief asks for at least three models each with tuning, so we tune **all four**, with two
methods. Both score every candidate with the same cross-validation and the same macro F1 as
section 5, so every number in this notebook stays comparable:

| Method | How it searches | Good for |
|---|---|---|
| **GridSearchCV** | Tries every combination in a list we write | Few settings, values we can name in advance |
| **Hyperopt (TPE)** | Learns from the trials it already ran and samples where the score looks promising | Many settings, or continuous ranges |

Each tuned pipeline is also scored **once** on the held-out test set. We report that number
next to the CV score, but the champion is picked by the CV score only (see section 7).

> **Note:** The brief asks for tuning on at least three models, so we tune all four. Without that rule we might have skipped Random Forest and K-Nearest Neighbors, because they were furthest behind before tuning. The results below show that would have been a mistake for Random Forest: it gains the most of all four models from tuning (about +0.04 macro F1) and ends up close to Logistic Regression.

### 6.1 Grid search

`GridSearchCV` scores **every** combination in the grid, so it is exhaustive — but the cost multiplies. Two settings with 2 and 4 values is already 8 combinations × 5 folds = 40 fits.

Note the `classifier__` prefix in the parameter names. It means *"the `C` of the pipeline step called `classifier`"*. That is how we reach inside the pipeline. Since the whole pipeline goes into the search, the preprocessor is refit inside every fold — so nothing leaks from the validation fold into the scaling.

In [ ]:
HTML GRID KOMPAKT + BATHCHART + SCHNELL GRÜN LANGSAM ROT

### 6.2 Hyperopt

The same four models, but now with **continuous** ranges: `learning_rate` anywhere between
0.02 and 0.3, not just the two values the grid could afford. Hyperopt tries a setting, sees
the score, and uses that to decide what to try next — so it covers a much wider space in the
same number of fits.

Three things to know about the syntax:

- `hp.loguniform` samples across orders of magnitude (0.01, 0.1, 1, 10 equally likely)
- `scope.int(...)` is needed because `hp.quniform` returns a float, and `n_estimators` must be a whole number
- Hyperopt **minimises**, so the objective returns `-score`

In [ ]:
HTML HYPEROPT KOMPAKT + BATHCHART + SCHNELL GRÜN LANGSAM ROT

## 7. Results — the comparison and the winner

The best tuned run of each model, shown as one card per model: the best settings the search found, how long it
took, the cross-validation score it was selected by, and its score on the held-out test set.

The **champion is the run with the highest cross-validated macro F1** — the metric we committed to in section 3. The test score is reported alongside it as an independent check, not as the selection criterion (choosing by the test score would spend the one honest number we have).

In [ ]:
HTML GEWINNER UND WARUM

In [ ]:
### Confusion matrix: each row shows what share of that true class went where
Basis + Grid + Hyperpod

In [ ]:
### The full leaderboard

Every run, including the untuned baselines, so the effect of tuning is visible.

## 8. The champion on the test set

The champion was chosen with cross-validation on the training data. Now we look at how it
behaves on the 20% it has never seen — per class, which is where the interesting part is.

## 9. Fit the champion on all the data and save it with joblib

Cross-validation on the training set showed which pipeline performs best, and the held-out test set confirmed that result.
Since the test score has already been measured, the test rows aren't needed for evaluation anymore. So for the final model we refit the champion pipeline on
the full dataset (training and test data combined), so it learns from every row we have before it goes into `app.py`.

We save the **complete pipeline** with `joblib`, not just the classifier. The pipeline contains
the preprocessing steps (imputation, scaling and one-hot encoding) and the tuned classifier.
When the app passes in a raw row of user answers, the pipeline transforms it exactly the same
way as the training data and then makes the prediction.

> The `models/` folder is listed in `.gitignore`, so it isn't committed to the repository.
> Running this notebook creates the model file again. Every step that involves randomness uses `random_state=42`, so you get the same model every time.

## 11. Final decision

The closing paragraph the project brief asks for. The numbers are filled in from the results
above, so the text always matches the latest run, even if the cleaned data changes.

In [ ]:
# ---- 1. Collect the numbers from the results above ----
untuned = all_runs[all_runs["tuning"] == "none"].sort_values("cv_f1_macro", ascending=False)
champ_runs = all_runs[(all_runs["model"] == CHAMPION_NAME) & (all_runs["tuning"] != "none")]
grid_run = champ_runs[champ_runs["tuning"] == "grid search"].iloc[0]
hyperopt_run = champ_runs[champ_runs["tuning"] == "hyperopt"].iloc[0]

untuned_champ = untuned[untuned["model"] == CHAMPION_NAME].iloc[0]
gain = CHAMPION_CV - untuned_champ["cv_f1_macro"]

# Undercontroller recall: untuned model (out-of-fold, section 5) vs. tuned champion (test set, section 8)
if best_name == CHAMPION_NAME:
    y_pred_untuned = y_pred_cv
else:
    y_pred_untuned = cross_val_predict(pipelines[CHAMPION_NAME], X_train, y_train, cv=cv)
recall_before = recall_score(y_train, y_pred_untuned, labels=["Undercontroller"], average=None)[0]
recall_after = recall_score(y_test, y_pred_test, labels=["Undercontroller"], average=None)[0]


# ---- 2. Build the two tables ----
def show(value):
    """Short readable version of a parameter value."""
    return f"{value:.3g}" if isinstance(value, float) else str(value)

model_rows = "\n".join(
    f"| **{row.model}** | **{row.cv_f1_macro:.3f}** |" if row.model == best_name
    else f"| {row.model} | {row.cv_f1_macro:.3f} |"
    for row in untuned.itertuples()
)

grid_values = param_grids[CHAMPION_NAME]
n_combinations = int(np.prod([len(v) for v in grid_values.values()]))   # size of the grid
defaults = pipelines[CHAMPION_NAME].named_steps["classifier"].get_params()
param_names = list(dict.fromkeys(list(grid_run["params"]) + list(hyperopt_run["params"])))
param_rows = []
for key in param_names:
    short = key.replace("classifier__", "")
    tried = (", ".join(show(v) for v in grid_values[key]) if key in grid_values
             else f"not searched (default {show(defaults[short])})")
    best_grid = show(grid_run["params"].get(key, defaults[short]))
    best_hyperopt = show(hyperopt_run["params"].get(key, defaults[short]))
    param_rows.append(f"| `{short}` | {tried} | {best_grid} | {best_hyperopt} |")
param_rows = "\n".join(param_rows)

if CHAMPION_TUNING == "grid search":
    winner_sentence = (f"Grid search had the higher CV score ({grid_run.cv_f1_macro:.4f} vs. "
                       f"{hyperopt_run.cv_f1_macro:.4f} for Hyperopt)")
else:
    winner_sentence = (f"Hyperopt had the higher CV score ({hyperopt_run.cv_f1_macro:.4f} vs. "
                       f"{grid_run.cv_f1_macro:.4f} for grid search)")

if champion_params.get("classifier__class_weight") == "balanced":
    class_weight_sentence = (
        f"`class_weight='balanced'` raised the recall of the smallest class, Undercontroller, from "
        f"{recall_before:.2f} (untuned, cross-validation) to {recall_after:.2f} (tuned, test set), "
        f"at the cost of a little overall accuracy ({untuned_champ.cv_accuracy:.1%} → {champion_test_acc:.1%})."
    )
else:
    class_weight_sentence = (
        f"the recall of the smallest class, Undercontroller, went from {recall_before:.2f} "
        f"(untuned, cross-validation) to {recall_after:.2f} (tuned, test set)."
    )


# ---- 3. The closing paragraph ----
final_decision = f"""
**Baseline comparison.** We first scored all four models with their default settings, using 
{N_SPLITS}-fold stratified cross-validation on the training set. {best_name} showed the strongest initial performance:

| Model | CV macro F1 (untuned) |
|---|---|
{model_rows}

**Hyperparameter tuning.** We then tuned every model twice: with GridSearchCV
({n_combinations} combinations for {CHAMPION_NAME})
and with Hyperopt ({MAX_EVALS} trials over wider, continuous ranges, see section 6.2).
For {CHAMPION_NAME}:

| Hyperparameter | Values tried in grid search | Best (grid search) | Best (Hyperopt) |
|---|---|---|---|
{param_rows}

{winner_sentence}, so the winning configuration is {format_params(champion_params)}.

**Result of the final model:**

- Cross-validated macro F1 (training set): **{CHAMPION_CV:.3f}**
- Macro F1 on the held-out 20% test set: **{CHAMPION_TEST:.3f}**
- Accuracy on the test set: **{champion_test_acc:.1%}**
- Baseline that always predicts "Moderate": {baseline_f1:.3f} macro F1

**Compared to the default parameters.** The overall gain in macro F1 is
{'small' if abs(gain) < 0.02 else 'clear'} ({gain:+.3f}, from {untuned_champ.cv_f1_macro:.3f} to {CHAMPION_CV:.3f}), but
{class_weight_sentence} Undercontroller is still the hardest class, because only two of the 19
statements (C4 and A4) measure the two traits that define it (see section 5). The
cross-validation and test scores are almost the same ({CHAMPION_CV:.3f} vs. {CHAMPION_TEST:.3f}),
so the model isn't overfitting the training data and generalises well to new data.

**Decision.** This is why we selected this pipeline (preprocessing + {CHAMPION_NAME}) as the final model, refit it on the full dataset and saved it with joblib as
`models/personality_pipeline.joblib`. The Streamlit app (`streamlit run app.py`) loads this file and serves the predictions.
"""

display(Markdown(final_decision))cd C:\Users\mwern\Downloads\ocean_personality_app